# 01 - Building a Knowledge Base

## 🎯 Learning Objectives

By the end of this notebook, you will:
- Load documents from various sources using document loaders
- Split text into optimized chunks using text splitters
- Generate embeddings with SiliconFlow's BGE-M3 model
- Store vectors in OceanBase with metadata
- Perform similarity search and retrieval

## 📚 Knowledge Base Pipeline

A typical retrieval workflow:

```plain
Sources → Document Loaders → Documents → Text Splitters → Chunks → Embeddings → Vector Store → Retriever
```

Each component is modular and can be swapped without rewriting the application logic.

## 🔧 Setup

First, let's set up our environment and models.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv("../.env")

# Verify configuration
print("✅ Configuration loaded:")
print(f"📍 OceanBase: {os.getenv('OCEANBASE_HOST')}:{os.getenv('OCEANBASE_PORT')}")
print(f"📍 Database: {os.getenv('OCEANBASE_DB')}")
print(f"📍 Embedding Model: {os.getenv('SILICONFLOW_EMBEDDING_MODEL', 'BAAI/bge-m3')}")

✅ Configuration loaded:
📍 OceanBase: 127.0.0.1:2881
📍 Database: test
📍 Embedding Model: BAAI/bge-m3


In [2]:
from langchain_dev_utils.embeddings import register_embeddings_provider, load_embeddings

# Register SiliconFlow embeddings provider
SILICONFLOW_BASE_URL = os.getenv("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1")

register_embeddings_provider(
    provider_name="siliconflow",  # Fixed: use provider_name instead of provider
    embeddings_model="openai-compatible",
    base_url=SILICONFLOW_BASE_URL,
)

# Load embedding model
EMBEDDING_MODEL_NAME = os.getenv("SILICONFLOW_EMBEDDING_MODEL", "BAAI/bge-m3")
embeddings = load_embeddings(f"siliconflow:{EMBEDDING_MODEL_NAME}")

print(f"✅ Loaded embedding model: {EMBEDDING_MODEL_NAME}")

✅ Loaded embedding model: BAAI/bge-m3


In [3]:
# OceanBase connection parameters
connection_args = {
    "host": os.getenv("OCEANBASE_HOST", "127.0.0.1"),
    "port": int(os.getenv("OCEANBASE_PORT", "2881")),
    "user": os.getenv("OCEANBASE_USER", "root@test"),
    "password": os.getenv("OCEANBASE_PASSWORD", ""),
    "db_name": os.getenv("OCEANBASE_DB", "test"),
}

print("✅ OceanBase connection configured")

✅ OceanBase connection configured


## 📄 Step 1: Document Loaders

Document loaders ingest data from external sources and return standardized `Document` objects.

### Loading PDF Documents

We'll load Nike's 10-K annual report (2023) - a real-world financial document with rich content.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

# Load Nike 10-K PDF (located in notebooks/ parent directory)
pdf_path = "./data/nke-10k-2023.pdf"

print(f"📄 Loading PDF: {pdf_path}")
loader = PyPDFLoader(pdf_path)

# Load all pages
documents = loader.load()

print(f"✅ Loaded {len(documents)} pages from PDF")
print(f"\n📊 Document statistics:")
total_chars = sum(len(doc.page_content) for doc in documents)
print(f"   Total characters: {total_chars:,}")
print(f"   Average page length: {total_chars // len(documents):,} characters")

# Show sample document
print(f"\n📄 Sample page (page 1):")
print(f"   Metadata: {documents[0].metadata}")
print(f"   Content preview: {documents[0].page_content[:200]}...")

# Show another sample from middle of document
mid_page = len(documents) // 2
print(f"\n📄 Sample page (page {mid_page + 1}):")
print(f"   Metadata: {documents[mid_page].metadata}")
print(f"   Content preview: {documents[mid_page].page_content[:200]}...")

📄 Loading PDF: ./data/nke-10k-2023.pdf
✅ Loaded 107 pages from PDF

📊 Document statistics:
   Total characters: 376,344
   Average page length: 3,517 characters

📄 Sample page (page 1):
   Metadata: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': './data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}
   Content preview: Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑  ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
F...

📄 Sample page (page 54):
   Metadata: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40

## ✂️ Step 2: Text Splitters

Text splitters break large documents into smaller chunks for better retrieval.

### RecursiveCharacterTextSplitter

For PDF documents, we use larger chunks to preserve context across page boundaries.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize text splitter with parameters optimized for PDF content
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Larger chunks for comprehensive context
    chunk_overlap=100,  # More overlap to maintain continuity
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Try to split on these in order
)

# Split documents
splits = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} pages into {len(splits)} chunks")
print(f"\n📊 Chunk statistics:")
chunk_lengths = [len(split.page_content) for split in splits]
print(f"   Average chunk size: {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
print(f"   Min: {min(chunk_lengths)} | Max: {max(chunk_lengths)} characters")

# Show sample chunks
print(f"\n📄 Sample chunks:")
for i, split in enumerate(splits[:2], 1):
    print(f"\nChunk {i}:")
    print(f"   Content: {split.page_content[:200].replace(chr(10), ' ')}...")
    print(f"   Metadata: {split.metadata}")

✅ Split 107 pages into 460 chunks

📊 Chunk statistics:
   Average chunk size: 829 characters
   Min: 54 | Max: 999 characters

📄 Sample chunks:

Chunk 1:
   Content: Table of Contents UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) ☑  ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934 F...
   Metadata: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': './data/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}

Chunk 2:
   Content: (Title of each class) (Trading symbol) (Name of each exchange on which registered) SECURITIES REGISTERED

## 🗄️ Step 3: Store in Vector Database

Now let's create embeddings and store them in OceanBase with HNSW indexing for efficient retrieval.

In [6]:
from langchain_oceanbase.vectorstores import OceanbaseVectorStore

# Create vector store with HNSW index for efficient similarity search
vector_store = OceanbaseVectorStore(
    embedding_function=embeddings,
    table_name="langchain_knowledge_base",
    connection_args=connection_args,
    vidx_metric_type="cosine",  # Use cosine similarity
    index_type="HNSW",  # Hierarchical Navigable Small World index
    vidx_algo_params={"M": 16, "efConstruction": 200},  # HNSW parameters
    drop_old=True,  # Start fresh
    normalize=True,  # Normalize vectors for cosine similarity
)

print("✅ Vector store initialized")
print(f"📊 Table: langchain_knowledge_base")
print(f"📐 Metric: cosine similarity")
print(f"🔍 Index: HNSW (M=16, efConstruction=200)")

✅ Vector store initialized
📊 Table: langchain_knowledge_base
📐 Metric: cosine similarity
🔍 Index: HNSW (M=16, efConstruction=200)


In [7]:
# Add documents to vector store in batches
# SiliconFlow API has a batch size limit of 64 for embeddings
print("⏳ Generating embeddings and storing in OceanBase...")
print("   (This may take a moment depending on the number of chunks)")

BATCH_SIZE = 50  # Use 50 to stay safely under the 64 limit
all_ids = []

for i in range(0, len(splits), BATCH_SIZE):
    batch = splits[i:i + BATCH_SIZE]
    batch_ids = vector_store.add_documents(batch)
    all_ids.extend(batch_ids)
    print(f"   Processed {min(i + BATCH_SIZE, len(splits))}/{len(splits)} chunks...")

ids = all_ids

print(f"\n✅ Successfully added {len(ids)} document chunks to vector store")
print(f"🆔 Sample IDs: {ids[:3]}")

⏳ Generating embeddings and storing in OceanBase...
   (This may take a moment depending on the number of chunks)
   Processed 50/460 chunks...
   Processed 100/460 chunks...
   Processed 150/460 chunks...
   Processed 200/460 chunks...
   Processed 250/460 chunks...
   Processed 300/460 chunks...
   Processed 350/460 chunks...
   Processed 400/460 chunks...
   Processed 450/460 chunks...
   Processed 460/460 chunks...

✅ Successfully added 460 document chunks to vector store
🆔 Sample IDs: ['54a92dc7-7512-4e8b-9b2b-1a4f20df91c6', '6421ca93-11b3-4687-8c2c-5f98b4389062', 'd27565d1-c7b8-4c63-862b-213ec300c2af']


## 🔍 Step 4: Retrieval - Similarity Search

Now that we have a knowledge base, let's perform similarity searches.

### Basic Similarity Search

In [8]:
# Simple similarity search
query = "What are Nike's main revenue sources and business segments?"

results = vector_store.similarity_search(query, k=3)

print(f"🔍 Query: '{query}'")
print(f"\n📋 Top {len(results)} results:\n")

for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"   Content: {doc.page_content[:200].replace(chr(10), ' ')}...")
    print(f"   Metadata: {doc.metadata}")
    print()

🔍 Query: 'What are Nike's main revenue sources and business segments?'

📋 Top 3 results:

Result 1:
   Content: Table of Contents NOTE 1 — SUMMARY OF SIGNIFICANT ACCOUNTING POLICIES DESCRIPTION OF BUSINESS NIKE, Inc. is a worldwide leader in the design, development and worldwide marketing and selling of athleti...
   Metadata: {'page': 63, 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'source': './data/nke-10k-2023.pdf', 'creator': 'EDGAR Filing HTML Converter', 'moddate': '2023-07-20T16:22:08-04:00', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'page_label': '64', 'total_pages': 107, 'creationdate': '2023-07-20T16:22:00-04:00'}

Result 2:
   Content: Table of Contents ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OFFINANCIAL CONDITION AND RESULTS OF OPERATIONS OVERVIEW NIKE designs, develops, markets and

### Similarity Search with Scores

Get similarity scores to understand relevance.

In [9]:
# Simple similarity search
query = "What are Nike's main business segments and revenue sources?"

results = vector_store.similarity_search(query, k=3)

print(f"🔍 Query: '{query}'")
print(f"\n📋 Top {len(results)} results:\n")

for i, doc in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"   Content: {doc.page_content[:200].replace(chr(10), ' ')}...")
    print(f"   Metadata: {doc.metadata}")
    print()

🔍 Query: 'What are Nike's main business segments and revenue sources?'

📋 Top 3 results:

Result 1:
   Content: Table of Contents NOTE 1 — SUMMARY OF SIGNIFICANT ACCOUNTING POLICIES DESCRIPTION OF BUSINESS NIKE, Inc. is a worldwide leader in the design, development and worldwide marketing and selling of athleti...
   Metadata: {'page': 63, 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'source': './data/nke-10k-2023.pdf', 'creator': 'EDGAR Filing HTML Converter', 'moddate': '2023-07-20T16:22:08-04:00', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'page_label': '64', 'total_pages': 107, 'creationdate': '2023-07-20T16:22:00-04:00'}

Result 2:
   Content: Table of Contents ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OFFINANCIAL CONDITION AND RESULTS OF OPERATIONS OVERVIEW NIKE designs, develops, markets and

### Search with Metadata Filtering

Filter results by metadata for more targeted retrieval.

In [10]:
# Search with relevance scores
query = "What were Nike's total revenues in 2023?"

results_with_scores = vector_store.similarity_search_with_score(query, k=3)

print(f"🔍 Query: '{query}'")
print(f"\n📊 Results with similarity scores:\n")

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"Result {i} (Distance: {score:.4f}):")
    print(f"   Content: {doc.page_content[:150].replace(chr(10), ' ')}...")
    print(f"   Page: {doc.metadata.get('page', 'N/A')}")
    print()

🔍 Query: 'What were Nike's total revenues in 2023?'

📊 Results with similarity scores:

Result 1 (Distance: 0.2844):
   Content: speed and responsiveness as we serve consumers globally. FINANCIAL HIGHLIGHTS • In fiscal 2023, NIKE, Inc. achieved record Revenues of $51.2 billion, ...
   Page: 30

Result 2 (Distance: 0.3102):
   Content: Table of Contents FISCAL 2023 NIKE BRAND REVENUE HIGHLIGHTSThe following tables present NIKE Brand revenues disaggregated by reportable operating segm...
   Page: 35

Result 3 (Distance: 0.3113):
   Content: Table of Contents YEAR ENDED MAY 31, (Dollars in millions) 2023 2022 2021 REVENUES North America $ 21,608 $ 18,353 $ 17,179  Europe, Middle East & Afr...
   Page: 88



## 🎯 Step 5: Create a Retriever

A retriever provides a standard interface for retrieving documents given a query.

In [11]:
# Search with metadata filter
query = "risk factors"

# Only search documents from early pages (typically where risk factors are disclosed)
# Note: Metadata filtering by page number may not work with all vector stores
# This demonstrates the concept
try:
    filtered_results = vector_store.similarity_search(
        query,
        k=3,
        # Note: Page filtering depends on vector store implementation
    )
    
    print(f"🔍 Query: '{query}'")
    print(f"\n📋 Search results:\n")
    
    for i, doc in enumerate(filtered_results, 1):
        print(f"Result {i}:")
        print(f"   Content: {doc.page_content[:200].replace(chr(10), ' ')}...")
        print(f"   Page: {doc.metadata.get('page', 'N/A')}")
        print()
except Exception as e:
    print(f"⚠️ Metadata filtering not fully supported: {e}")
    print("Showing standard similarity search results instead:")
    
    filtered_results = vector_store.similarity_search(query, k=3)
    for i, doc in enumerate(filtered_results, 1):
        print(f"\nResult {i}:")
        print(f"   Content: {doc.page_content[:200].replace(chr(10), ' ')}...")
        print(f"   Page: {doc.metadata.get('page', 'N/A')}")

🔍 Query: 'risk factors'

📋 Search results:

Result 1:
   Content: availability, safety and efficacy of vaccines, including against emerging variants of the infectious disease, and global economic conditions. Additionally, disruptions have in the past made it more ch...
   Page: 15

Result 2:
   Content: confidence; • Cancellation or postponement of sports seasons and sporting events in multiple countries, and bans on large public gatherings, which have reduced and in the future could reduce consumer ...
   Page: 15

Result 3:
   Content: manufacturers were able to source sufficient quantities of raw materials for use in our footwear and apparel products. Refer to Item 1A. Risk Factors, for additional discussion of the impact of sourci...
   Page: 6



## 📊 Knowledge Base Statistics

In [12]:
# Create retriever from vector store
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 results
)

# Test retriever
query = "What are Nike's key strategic initiatives and growth opportunities?"
retrieved_docs = retriever.invoke(query)

print(f"🔍 Query: '{query}'")
print(f"\n📚 Retrieved {len(retrieved_docs)} documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(f"   Content: {doc.page_content[:200].replace(chr(10), ' ')}...")
    print(f"   Page: {doc.metadata.get('page', 'N/A')}")
    print()

🔍 Query: 'What are Nike's key strategic initiatives and growth opportunities?'

📚 Retrieved 3 documents:

Document 1:
   Content: Table of Contents ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OFFINANCIAL CONDITION AND RESULTS OF OPERATIONS OVERVIEW NIKE designs, develops, markets and sells athletic footwear, apparel, equipment,...
   Page: 30

Document 2:
   Content: representation by works councils (which may be entitled to information and consultation on certain subsidiary decisions) or by organizations similar to a union. In certain European countries, we are r...
   Page: 8

Document 3:
   Content: the demand for our products. We must, therefore, respond to trends and shifts in consumer preferences by adjusting the mix of existing product offerings, developing new products, styles and categories...
   Page: 4



## 🎉 Summary

Congratulations! You've successfully built a knowledge base with real-world data!

### What we accomplished:

- ✅ **Document Loading**: Loaded Nike's 10-K annual report (2023) using PyPDFLoader
- ✅ **Text Splitting**: Split PDF pages into optimized chunks (1000 chars with 100 char overlap)
- ✅ **Embeddings**: Generated 1024-dimensional vectors using SiliconFlow's BGE-M3 model
- ✅ **Vector Storage**: Stored embeddings in OceanBase with HNSW indexing for fast retrieval
- ✅ **Retrieval**: Performed semantic similarity search on financial data
- ✅ **Retriever Interface**: Created a reusable retriever for downstream RAG applications

### Key Concepts:

1. **PDF Loading**: PyPDFLoader extracts text page-by-page with metadata
2. **Semantic Search**: Find relevant information using meaning, not just keywords
3. **Chunk Optimization**: Balance between context (larger chunks) and precision (smaller chunks)
4. **HNSW Indexing**: Efficient approximate nearest neighbor search for large datasets
5. **Real-world Data**: Financial reports contain complex structured and unstructured information

### Knowledge Base Details:

- **Source**: Nike 10-K 2023 Annual Report (~2.3MB PDF)
- **Content**: Financial statements, risk factors, business operations, strategy
- **Embeddings**: BAAI/bge-m3 (1024 dimensions, multilingual)
- **Storage**: OceanBase with cosine similarity and HNSW index

### Next Steps:

In **Notebook 02**, we'll implement **2-Step RAG**:
- Use this knowledge base to answer questions about Nike
- Build a Q&A system that retrieves relevant context
- Generate accurate, grounded answers from the 10-K report
- Compare responses with and without RAG

In [13]:
# Gather statistics about our knowledge base
print("📊 Knowledge Base Statistics:")
print(f"\n📄 Total chunks stored: {len(ids)}")
print(f"📄 Original pages: {len(documents)}")
print(f"📊 Chunks per page (avg): {len(ids) / len(documents):.1f}")

# Get embedding dimension
embedding_dim = len(embeddings.embed_query('test'))

print(f"\n🗄️ Database: {connection_args['db_name']}")
print(f"📋 Table: langchain_knowledge_base")
print(f"🔢 Embedding dimension: {embedding_dim}")
print(f"📐 Similarity metric: cosine")
print(f"🔍 Index type: HNSW")

print(f"\n💾 Storage estimate:")
total_vectors_size_mb = (len(ids) * embedding_dim * 4) / (1024 * 1024)  # 4 bytes per float
print(f"   Vector data: ~{total_vectors_size_mb:.1f} MB")

📊 Knowledge Base Statistics:

📄 Total chunks stored: 460
📄 Original pages: 107
📊 Chunks per page (avg): 4.3

🗄️ Database: test
📋 Table: langchain_knowledge_base
🔢 Embedding dimension: 1024
📐 Similarity metric: cosine
🔍 Index type: HNSW

💾 Storage estimate:
   Vector data: ~1.8 MB


## 💡 Note on Knowledge Base Persistence

The knowledge base we created is **persisted in OceanBase** and will be available for the next notebooks (03, 04, 05).

You can verify the data is stored by:
- Checking the SeekDB dashboard
- Running similarity searches in this notebook
- The table `langchain_knowledge_base` contains all the embeddings

**No cleanup needed** - we'll use this knowledge base in subsequent notebooks for RAG implementations!

## 💡 Additional Resources

- [LangChain Document Loaders](https://python.langchain.com/docs/integrations/document_loaders/)
- [LangChain Text Splitters](https://python.langchain.com/docs/integrations/text_splitters/)
- [OceanBase Vector Store Documentation](https://github.com/oceanbase/langchain-oceanbase)
- [BGE-M3 Embedding Model](https://huggingface.co/BAAI/bge-m3)
- [HNSW Algorithm Paper](https://arxiv.org/abs/1603.09320)
- [PyPDFLoader Documentation](https://python.langchain.com/docs/integrations/document_loaders/pypdf/)